In [3]:
import numpy as np
from numba import cuda
import math

@cuda.jit
def histogram_kernel(d_data, d_hist, num_bins):
    """
    Each thread reads an element from the input array,
    calculates its corresponding bin, and safely increments that bin.
    """
    # Calculate unique global thread ID
    idx = cuda.grid(1)

    # Boundary check to make sure thread isn't reading past array size
    if idx < d_data.size:
        val = d_data[idx]

        # Ensure the value falls within the expected bin boundaries [0, num_bins-1]
        if 0 <= val < num_bins:
            # Cast to integer to use as an array index
            bin_idx = int(val)

            # CRITICAL: Use atomic add to prevent race conditions
            # across threads writing to the same bin simultaneously.
            cuda.atomic.add(d_hist, bin_idx, 1)

def parallel_histogram(h_data, num_bins):
    n = h_data.size

    # Allocate device memory and copy input array
    d_data = cuda.to_device(h_data.astype(np.int32))

    # Initialize the histogram bin array on the GPU with zeros
    d_hist = cuda.device_array(num_bins, dtype=np.int32)
    d_hist.bind() # Clear/zero out memory on allocation

    # Setup grid layout
    threads_per_block = 256
    blocks_per_grid = math.ceil(n / threads_per_block)

    # Launch kernel
    histogram_kernel[blocks_per_grid, threads_per_block](d_data, d_hist, num_bins)

    # Copy calculated bins back to host memory
    return d_hist.copy_to_host()

# -------------------------------------------------------------------
# Verification & Host Execution
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Parameters
    array_size = 1_000_000
    num_bins = 10  # We expect values ranging from 0 up to 9

    # Generate random integer data between [0, 9]
    np.random.seed(42)
    test_data = np.random.randint(0, num_bins, size=array_size).astype(np.int32)

    print(f"Generated {array_size:,} random elements bounded between 0 and {num_bins-1}.")
    print("-" * 60)

    # Run GPU Histogram Calculation
    gpu_histogram = parallel_histogram(test_data, num_bins)

    # Run CPU NumPy baseline calculation for verification
    # np.bincount counts occurrences of each non-negative integer
    cpu_histogram = np.bincount(test_data, minlength=num_bins)

    # Print Results for comparison
    print(f"{'Bin Index':<12}{'GPU Count':<15}{'CPU Count':<15}")
    print("-" * 60)
    for i in range(num_bins):
        print(f"{i:<12}{gpu_histogram[i]:<15}{cpu_histogram[i]:<15}")
    print("-" * 60)

    # Verify correctness
    if np.array_equal(gpu_histogram, cpu_histogram):
        print("Success! The GPU histogram matches the CPU baseline precisely.")
    else:
        print("Mismatch! The histogram calculation was incorrect.")

Generated 1,000,000 random elements bounded between 0 and 9.
------------------------------------------------------------
Bin Index   GPU Count      CPU Count      
------------------------------------------------------------
0           100114         100114         
1           100049         100049         
2           100255         100255         
3           99977          99977          
4           100482         100482         
5           99940          99940          
6           99839          99839          
7           99679          99679          
8           99783          99783          
9           99882          99882          
------------------------------------------------------------
Success! The GPU histogram matches the CPU baseline precisely.
